# Lecture 10 - Root finding in multiple dimensions, and project pitch formation

Last lecture, we discussed some techniques one can use to solve an equation for one variable. This lecture, we will generalize these techniques to equations with multiple variables:
$$
{\bf f}({\bf x}) = 0,
$$
where both ${\bf f}$ and ${\bf x}$ are multi-dimensional vectors. To carry out this task, we will merge the techniques that we learned when solving linear systems, with those solving general non-linear equations of one variable.

After introducing multi-dimensional root finding methods, we will have a bit of a "social time," where all students will finalize
1. Your project groups
2. Your project pitch topics
Next Thursday, all students will submit a short paragraph describing their project, and their proposed project groups.

### Accepting and submitting

To accept this assignment on Classroom 50 with your GitHub profile enrolled in the class, click on the link \
https://classroom50.org/PsuAstro410/410astro26/assignments/lecture-10/accept \
and follow the steps the webpage prompts.

To submit this assignment, follow the steps in "First time Classroom 50 setup" and "Accepting and Submitting Assignments" on the course webpage:\
https://psuastro410.github.io/tips/github/

### Resources and Acknowledgements

This lecture made use of material from the following sources:

In [ ]:
# Nice things to have for this lecture

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import sympy

# Multivariate Root Finding

Imagine a vector function,

$${\bf f}({\bf x}) = \left ( \begin{array}{c} f_1({\bf x}) \\ f_2({\bf x}) \\ \vdots \\ f_N({\bf x}) \end{array} \right )$$

where

$${\bf x} = \left ( \begin{array}{c} x_1 \\ x_2 \\ \vdots \\ x_N \end{array} \right )$$

We want to find the ${\bf x}$ that zeros ${\bf f}({\bf x})$, i.e.,

$${\bf f}({\bf x}) = 0$$

We'll use a generalization of Newton's method to systems of equations.

**Note:**
This is the simplest case of a multivariable root finding algorithm.  More sophisticated methods exist, but because the goal of this course is to survey the field of computational astrophysics, the method described in this lecture will give you a feel for what kind of manipulations are typically involved.

## Solution procedure

Start with an initial guess, ${\bf x}^{(0)}$

Taylor expand our function seeking a correction $\delta {\bf x}$.  Each component has the form:
  
$$f_i ({\bf x}^{(0)} + \delta {\bf x}) = f_i({\bf x}^{(0)}) + \sum_{j=1}^N \frac{\partial f_i}{\partial x_j} \delta x_j + \ldots$$
  
and we can write the vector form as:
  
$$0 = {\bf f}({\bf x}^{(0)} + \delta {\bf x}) \approx {\bf f}({\bf x}^{(0)}) + {\bf J} \cdot \delta {\bf x}$$
  
where ${\bf J}$ is the [Jacobian](https://en.wikipedia.org/wiki/Jacobian_matrix_and_determinant):
  
$${\bf J} = \left ( \begin{array}{cccc} \frac{\partial f_1}{\partial x_1} &
                                        \frac{\partial f_1}{\partial x_2} &
                                        \cdots &
                                        \frac{\partial f_1}{\partial x_N} \\
%
                                        \frac{\partial f_2}{\partial x_1} &
                                        \frac{\partial f_2}{\partial x_2} &
                                        \cdots &
                                        \frac{\partial f_2}{\partial x_N} \\
%
                                        \vdots & \vdots & \ddots & \vdots \\
%
                                        \frac{\partial f_N}{\partial x_1} &
                                        \frac{\partial f_N}{\partial x_2} &
                                        \cdots &
                                        \frac{\partial f_N}{\partial x_N} \end{array} \right )$$


This can be expresses as the linear system:

$${\bf J} \cdot \delta {\bf x} = - {\bf f}({\bf x}^{(0)})$$

Our solution procedure is iterative:

* Iterate over $k$, seeking a new guess at the solution ${\bf x}^{k+1}$:

  * Solve the linear system:

    $${\bf J} \cdot \delta {\bf x} = - {\bf f}({\bf x}^{(k)})$$
  
  * Correct our initial guess:

    $${\bf x}^{(k+1)} = {\bf x}^{(k)} + \delta {\bf x}$$
  
  * Stop iteration if

    $$\| \delta {\bf x} \| < \epsilon \| {\bf x}^{(k)} \|$$
  
    where $\| \cdot \|$ is a suitable vector norm.

# Application: Equation of State Calculations

Equations of state (EOSs) are often formulated in terms of density, $\rho$,
and temperature, $T$.  But it is common to need to use the equation
of state with different inputs (like pressure, $p$) and recover the density 
or temperature that yields thermodynamic consistency.  This is a root-finding problem (sometimes we say that we need to *invert* the EOS).

Here we will consider an ideal gas and radiation EOS:

\begin{align*}
p &= \frac{\rho k T}{\mu m_u} + \frac{1}{3} a T^4 \\
e &= \frac{3}{2} \frac{k T}{\mu m_u} + \frac{a T^4}{\rho}
\end{align*}

where $e$ is the specific internal energy (energy / mass)
and the constants are:

* k: Boltzmann's constant ($k = 1.38\times 10^{-16}~\mathrm{erg/K}$)
* $\mu$: the mean molecular weight of the matter (we'll just take $\mu = 4$)
* $m_u$: the atomic mass unit ($m_u = 1.66\times 10^{-24}~\mathrm{g}$)
* $a$: the radiation constant ($a = 7.56\times 10^{-15}~\mathrm{erg/cm^3/K^4}$).

Imagine that we know the value of $p$, and $e$, which we'll call them $p_\star$ and $e_\star$.  We can find the density and temperature that give these by solving the system:

\begin{align*}
p(\rho, T) &= p_\star \\
e(\rho, T) &= e_\star
\end{align*}

Let's define $\Psi$ as:

$$\Psi = \left ( \begin{array}{c} p(\rho, T) - p_\star \\
                                  e(\rho, T) - e_\star \end{array} \right )$$

and imagine that we have an initial guess for density, $\rho_0$, and temperature, $T_0$.  We then want to find the corrections, $\delta \rho$
and $\delta T$, such that:

$$\Psi(\rho_0 + \delta\rho, T_0 + \delta T) = 0$$

Tayloring expanding, we get:

$$0 = \Psi(\rho_0 + \delta_\rho, T_0 + \delta T) = \Psi(\rho_0, T_0) + {\bf J} \cdot \left ( \begin{array}{c} \delta \rho \\ \delta T \end{array} \right ) + \ldots$$

where ${\bf J}$ is the Jacobian,

$${\bf J} = \left ( \begin{array}{cc} \partial p/\partial \rho |_T & \partial p/\partial T |_\rho \\ 
                                      \partial e/\partial \rho |_T & \partial e/\partial T |_\rho \end{array} \right )$$

We will calculate these partial derivatives analytically from the EOS, but using computer algebra.

We can then find the correction by solving the system:

$${\bf J} \cdot \left (\begin{array}{c} \delta \rho \\ \delta T \end{array} \right ) = - \Psi(\rho_0, T_0)$$

We iterate until the correction is small.

## Calculating partial derivatives using SymPy

First, we calculate the partial derivatives. For this application, there are nice analytic expressions describing the pressure and energy of the system. One can always calculate these analytically if one has enough patience, but 
1. It is easy to make mistakes when calculating these partial derivatives
2. Calculating these partial derivatives can be time consuming, expecially for non-trivial functions

Fortunately, there is symbolic mathematical packages in python! SymPy in particular is quite good at doing tedious mathematical manipulations, like algebra, derivatives, integrals, etc. 

First, we define the symbols, both constants and variables, for our calculation:

In [ ]:
rho_s, T_s, k_s, mu_s, m_u_s, a_s = sympy.symbols("rho T k mu m_u a", real=True)

Next, we define the symbolic expressions for both the pressure and energy:

In [ ]:
p_s = (rho_s*k_s*T_s)/(mu_s*m_u_s) + sympy.Rational(1, 3) * a_s*T_s**4
p_s

In [ ]:
e_s = sympy.Rational(3,2) * (k_s*T_s)/(mu_s*m_u_s) + a_s*T_s**4/rho_s
e_s

Looks good! Now lets get to work.

We can use `sympy.diff` to take the partial derivatives analytically, and calculate the Jacobian. We can also use `sympy.simplify` to tell SymPy to look for ways to make the calculated expression a bit simpler. Code to carry out both tasks and print the result is displayed below.

In [ ]:
dpdrho_s = sympy.diff(p_s, rho_s)
dpdT_s = sympy.diff(p_s, T_s)
dedrho_s = sympy.diff(e_s, rho_s)
dedT_s = sympy.diff(e_s, T_s)

J_s = sympy.Matrix([[dpdrho_s, dpdT_s],
                    [dedrho_s, dedT_s]])

J_s = sympy.simplify(J_s)

J_s

Look at that! No need for pencil and paper, all done already.

We can even get SymPy to convert these different expressions into functions, that inputs and outputs numpy arrays. Below is a code that does this, for inputs $\rho$ and $T$:

In [ ]:
# constants
k = 1.38e-16  # erg/K
a = 7.56e-15  # erg/cm^3/K^4
m_u = 1.66e-24  # g

# Substituting numerical values for constants (this makes the code below much faster)
p_eval = p_s.subs({k_s: k, a_s: a, m_u_s: m_u})
e_eval = e_s.subs({k_s: k, a_s: a, m_u_s: m_u})
dpdrho_eval = dpdrho_s.subs({k_s: k, a_s: a, m_u_s: m_u})
dpdT_eval = dpdT_s.subs({k_s: k, a_s: a, m_u_s: m_u})
dedrho_eval = dedrho_s.subs({k_s: k, a_s: a, m_u_s: m_u})
dedT_eval = dedT_s.subs({k_s: k, a_s: a, m_u_s: m_u})
J_eval = J_s.subs({k_s: k, a_s: a, m_u_s: m_u})

# Creating numpy functions for all the quantities
p_np = sympy.lambdify((rho_s, T_s, mu_s), p_eval, "numpy")
e_np = sympy.lambdify((rho_s, T_s, mu_s), e_eval, "numpy")
dpdrho_np = sympy.lambdify((rho_s, T_s, mu_s), dpdrho_eval, "numpy")
dpdT_np = sympy.lambdify((rho_s, T_s, mu_s), dpdT_eval, "numpy")
dedrho_np = sympy.lambdify((rho_s, T_s, mu_s), dedrho_eval, "numpy")
dedT_np = sympy.lambdify((rho_s, T_s, mu_s), dedT_eval, "numpy")
J_np = sympy.lambdify((rho_s, T_s, mu_s), J_eval, "numpy")

Now we have a function that spits out all the quantities that we need to make our Jacobian.

## Defining EOS class

Now that we have calculated the partial derivatives, we can create a class that calculates the important quantities. We initialize the class to store all the relevant thermodynamic properties: mean molecular weight, density, temperature, pressure, specific energy, and all the thermodynamic partial derivatives.

In [ ]:
class EOSState:
    def __init__(self, rho=None, T=None,
                 p=None, e=None,
                 dpdrho=None, dpdT=None,
                 dedrho=None, dedT=None, mu=4):
        self.mu = mu
        self.rho = rho
        self.T = T
        
        self.p = p
        self.e = e
        self.dpdrho = dpdrho
        self.dpdT = dpdT
        self.dedrho = dedrho
        self.dedT = dedT

We can later include the specifics for the ideal EOS, for the object that we want to use to invert our EOS.

In [ ]:
def eos(state):

    state.p = p_np(state.rho, state.T, state.mu)
    state.e = e_np(state.rho, state.T, state.mu)

    state.dpdrho = dpdrho_np(state.rho, state.T, state.mu)
    state.dpdT = dpdT_np(state.rho, state.T, state.mu)

    state.dedrho = dedrho_np(state.rho, state.T, state.mu)
    state.dedT = dedT_np(state.rho, state.T, state.mu)

## Plotting the EOS

Lets plot the results of this EOS on a contour plot. Typical ranges for solar-like star are roughly $10^{-7} - 10^2 \ {\rm g} \ {\rm cm}^{-3}$ for density, and roughly $10^{3} - 10^{8} \ {\rm K}$ for temperature

In [ ]:
# grid in the orbital plane
rhog = np.logspace(-7, 2)
Tg = np.logspace(3, 8)
Rho, Temp = np.meshgrid(rhog, Tg)

state_numpy = EOSState(rho=Rho, T=Temp)
eos(state_numpy)
P = state_numpy.p
E = state_numpy.e

levels_logP = np.linspace(np.log10(np.min(P)), np.log10(np.max(P)), 40)
levels_logE = np.linspace(np.log10(np.min(E)), np.log10(np.max(E)), 40)

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(7, 13))

cf_P = ax[0].contourf(Rho, Temp, np.log10(P), levels=levels_logP, cmap="viridis", extend="min")
ax[0].contour(Rho, Temp, np.log10(P), levels=levels_logP, colors="k", linewidths=0.35, alpha=0.45)

cb_P = fig.colorbar(cf_P, ax=ax[0])
cb_P.set_label(r"$\log_{10}(P)$")

cf_E = ax[1].contourf(Rho, Temp, np.log10(E), levels=levels_logE, cmap="plasma", extend="min")
ax[1].contour(Rho, Temp, np.log10(E), levels=levels_logE, colors="k", linewidths=0.35, alpha=0.45)

cb_E = fig.colorbar(cf_E, ax=ax[1])
cb_E.set_label(r"$\log_{10}(E)$")

for i in range(2):
    ax[i].set_xscale('log')
    ax[i].set_yscale('log')
    ax[i].set_xlabel(r"$\rho \ [{\rm g} \ {\rm cm}^{-3}]$")
    ax[i].set_ylabel(r"$T \ [{\rm K}]$")
    
fig.tight_layout()
plt.show()

## Inverting the EOS

Now here is our implementation, where we pass in the pressure and energy we want, and optionally a guess for $\rho_0$ and $T_0$:

In [ ]:
def rhoT_from_pe(p_in, e_in, *, rho0=1, T0=1, tol=1.e-5):
    state = EOSState(rho=rho0, T=T0)

    err = 1.e30
    while (err > tol):
        # get the current thermodynamics
        eos(state)

        # construct the Jacobian and Psi
        J = np.array([[state.dpdrho, state.dpdT],
                      [state.dedrho, state.dedT]])        
        psi = np.array([state.p - p_in, state.e - e_in])

        # solve for the corrections
        delta = np.linalg.solve(J, -psi)

        # update our guesses
        state.rho += delta[0]
        state.T += delta[1]

        # compute the error
        err = max(abs(delta[0]/state.rho), abs(delta[1]/state.T))

    return state.rho, state.T

Now lets test this out

In [ ]:
p_star = 2.3e10
e_star = 3.87e13
rho, T = rhoT_from_pe(p_star, e_star, rho0=1.e2, T0=1.e5)
(rho, T)

Now we can check how well we did by calling the EOS with the $\rho$ and $T$ we found and comparing to the pressure and energy we wanted.

In [ ]:
s_new = EOSState(rho=rho, T=T)
eos(s_new)
print(f"pressure: we found {s_new.p} and wanted {p_star}")
print(f"energy: we found {s_new.e} and wanted {e_star}")

Not bad!

# Exercises (10 pt)

Today's exercises will be more social, and the goal is to crystalize your idea for your course project.

### Exercise 1 (2 pt)

Using your overleaf account:\
https://www.overleaf.com/ \
Create a new project called "project pitch." Keep the default article template.

### Exercise 2 (2 pt)

Share the link in the markdown cell below, and click the option so that "anyone can view":

**Link here**


### Exercise 3 (2 pt)

Give your project a title, based on the work you want to do.

### Exercise 4 (2 pt)

Using LaTeX's itemize command:\
https://www.overleaf.com/learn/latex/Lists \
List a few course topics that your project incooperates, which can (and probably should) incooperate topics that we have not covered yet. As a reminder, the course syllabus is below:\
https://psuastro410.github.io/syllabus/ \
Make this list as a separate section in your overleaf document.

### Exercise 5 (2 pt)

Social time! Chat with other class members, and finalize one to two other "collaborators" in your project group. Create a separate section in your overleaf document, also use LaTeX's itemize, and list your collaborators.